In [0]:
"""
Phase 3: Embeddings
Generates vector embeddings for each chunk in both Silver chunking-strategy
tables using Databricks' hosted foundation embedding model.
"""

import mlflow.deployments

# Sanity check the endpoint with a single chunk 

client = mlflow.deployments.get_deploy_client("databricks")

def get_embedding(text):
    response = client.predict(
        endpoint="databricks-bge-large-en",
        inputs={"input": [text]}
    )
    return response["data"][0]["embedding"]

# Test on a single chunk first before running the full table
sample_text = spark.table("rag_pipeline.main.silver_chunks_fixed_size").limit(1).collect()[0]["chunk_text"]
test_embedding = get_embedding(sample_text)
print(len(test_embedding), test_embedding[:5])

1024 [-0.004550933837890625, 0.0068206787109375, 0.037750244140625, 0.0220794677734375, -0.005123138427734375]


In [0]:
import mlflow.deployments
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, FloatType

# Define the distributed batch embedding function 

@pandas_udf(ArrayType(FloatType()))
def embed_batch(texts: pd.Series) -> pd.Series:
    client = mlflow.deployments.get_deploy_client("databricks")
    embeddings = []
    # Batch requests to the endpoint rather than one-by-one, to reduce call overhead
    batch_size = 20
    text_list = texts.tolist()
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = client.predict(
            endpoint="databricks-bge-large-en",
            inputs={"input": batch}
        )
        embeddings.extend([item["embedding"] for item in response["data"]])
    return pd.Series(embeddings)

In [0]:
#  Embed the fixed-size chunks table 


fixed_df = spark.table("rag_pipeline.main.silver_chunks_fixed_size")
fixed_embedded_df = fixed_df.withColumn("embedding", embed_batch("chunk_text"))

fixed_embedded_df.write.format("delta").mode("overwrite") \
    .saveAsTable("rag_pipeline.main.silver_embeddings_fixed_size")

# Embed the section-aware chunks table

section_df = spark.table("rag_pipeline.main.silver_chunks_section_aware")
section_embedded_df = section_df.withColumn("embedding", embed_batch("chunk_text"))

section_embedded_df.write.format("delta").mode("overwrite") \
    .saveAsTable("rag_pipeline.main.silver_embeddings_section_aware")

In [0]:
print("Fixed-size embeddings:", spark.table("rag_pipeline.main.silver_embeddings_fixed_size").count())
print("Section-aware embeddings:", spark.table("rag_pipeline.main.silver_embeddings_section_aware").count())

display(spark.table("rag_pipeline.main.silver_embeddings_fixed_size").select("chunk_id", "company", "embedding").limit(3))

Fixed-size embeddings: 6457
Section-aware embeddings: 7197


chunk_id company embedding 0 AAPL List(-0.004550934, 0.0068206787, 0.037750244, 0.022079468, -0.0051231384, -0.026123047, 0.008651733, 0.012702942, 0.0259552, 0.05050659, 0.0067977905, 0.035461426, -0.007801056, -0.018936157, -0.020233154, 0.0049095154, -6.351471E-4, -0.007873535, 0.025238037, 9.7227097E-4, -0.019439697, 0.025115967, -0.06719971, 4.0221214E-4, -0.043548584, 0.042266846, -0.009437561, 0.0058670044, 0.088256836, 0.01828003, -0.035491943, -0.047729492, 0.03515625, -0.06311035, -0.013175964, -0.019226074, 0.035736084, -0.034362793, -0.0027198792, -0.021224976, -0.002243042, 0.009887695, 0.073913574, -0.012626648, -0.023529053, -0.020690918, -0.030456543, -0.016586304, 0.011062622, -0.026412964, 0.008392334, 0.017501831, 0.03933716, -0.017990112, 0.04675293, -0.035827637, -0.034820557, -0.0090789795, -0.021972656, 0.05178833, 0.0128479, -0.027938843, 0.0496521, -0.056793213, -0.006038666, 0.035186768, 0.01499176, 0.0044517517, 0.008041382, -0.02305603, -0.015464783, 0.022064209, -0.051330566, 0.003545761, -0.0025043488, -0.033966064, -0.0056533813, -0.013771057, -0.02746582, -0.007423401, -0.017044067, 0.0055007935, 0.016403198, -0.008331299, 0.0022563934, 8.034706E-4, 0.019683838, -0.013465881, 0.032165527, -0.012237549, 0.028305054, 0.02973938, -0.016998291, -0.03768921, 0.030822754, 0.05206299, -0.019805908, 0.02104187, -0.007633209, 0.014434814, 0.059753418, 0.0019874573, -0.052703857, 0.0075950623, -0.006511688, 0.017990112, 0.009529114, -0.018234253, -0.013954163, -0.026275635, 0.041259766, -0.014961243, -0.0033798218, -6.9761276E-4, -0.044128418, 0.029083252, -0.00869751, 0.028442383, -0.040924072, 0.0018520355, 0.014129639, 0.01576233, 0.02027893, -0.005569458, -0.0057373047, -0.009841919, -0.007873535, 0.06378174, -0.04699707, 0.04046631, 0.02658081, -0.01902771, -0.027618408, 0.018554688, 0.0070495605, 0.031951904, -0.026977539, -0.030776978, 0.049743652, -0.024475098, 0.015731812, 0.02722168, -0.04196167, 0.09674072, -6.991625E-5, 0.030899048, -0.020233154, 0.016281128, -0.022827148, -0.02456665, -0.0052719116, 0.037506104, -0.0033817291, 0.010749817, -0.008804321, 0.004337311, -0.04083252, 1.2946129E-4, 0.002904892, 0.027511597, -0.008239746, 0.034118652, -0.015335083, 0.032928467, -0.047027588, 0.041259766, -0.015449524, -0.0030136108, -0.05831909, -0.013748169, 0.040130615, -0.009140015, -0.030776978, -0.03173828, -0.02168274, 0.018508911, 0.054504395, 0.019302368, -0.012779236, -0.0049552917, -0.027145386, 0.023468018, 0.06939697, 0.025650024, -0.01966858, 0.04257202, -0.01739502, -0.014297485, -0.011169434, -0.076171875, 0.030532837, 0.037139893, 0.01977539, 0.036071777, -0.031036377, -0.020889282, 0.011505127, 0.009742737, 0.007633209, -0.014541626, -0.052368164, -0.01979065, 0.003917694, 0.038208008, -0.023773193, -0.015281677, 0.053100586, 0.049926758, -0.0021686554, -0.003862381, 0.025131226, 0.017684937, -0.021759033, -0.008255005, 0.01449585, -0.014404297, -0.015045166, 0.020187378, 0.05480957, -0.009735107, 0.03591919, 0.036956787, 0.01436615, 0.051239014, 0.01411438, -0.043304443, 0.02973938, 0.05392456, -0.031555176, 0.0093307495, 0.006137848, 0.013290405, 0.013694763, 0.037506104, 0.02645874, 0.016357422, 0.0501709, 0.05368042, 0.011680603, 0.018096924, 0.0056533813, -0.008544922, 0.038391113, -8.788109E-4, 0.036865234, 0.034179688, -0.030700684, 0.0011491776, 0.009315491, 0.009246826, 8.2969666E-4, 0.034576416, -0.002986908, 0.02406311, 0.0046920776, -0.01234436, -0.003967285, 0.049713135, -0.039520264, -0.022857666, 0.017166138, -0.020477295, 0.013999939, 0.045806885, -0.024093628, -0.012992859, 0.007598877, 0.004009247, -0.012207031, -0.030883789, -0.031219482, -0.029403687, -0.04901123, -0.022399902, -0.033843994, -0.027511597, 0.029525757, -0.02633667, 0.06021118, -0.05038452, 0.012008667, -0.004360199, 0.022644043, 0.048950195, -0.031143188, 0.002565384, -0.020446777, -0.0044174194, -0.048614502, 0.028625488, -0.036010742, 0.04095459, 0.01689148, -0.036468506, 0.02532959, -0.03601074